# Newton's law of cooling

*thermodynamics · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/thermodynamics/newton_cooling)

A row of hot cups of coffee cools towards room temperature. Each cup has its own starting temperature and its own cooling constant, yet all of them live in one tensor, so a single line of PyTorch advances every cup at once — and the result matches the exponential solution.

**Vector form**

$$\frac{dT}{dt} = -k\,(T - T_{\text{env}}),\qquad T(t) = T_{\text{env}} + (T_0 - T_{\text{env}})\,e^{-kt}$$

**Index (tensor) form**

$$\dot T_b = -k_b\,(T_b - T_{\text{env}}),\qquad T_{tb} = T_{\text{env}} + (T_{0,b} - T_{\text{env}})\,e^{-k_b t_t}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $T_b$ | `T` | (B,) | temperature of cup $b$ (°C) |
| $k_b$ | `k` | (B,) | cooling constant of cup $b$ (1/min): bigger means faster cooling |
| $T_{\text{env}}$ | `T_env` | scalar | room temperature, broadcast to every cup |
| $\dot T_b$ | `-k * (T - T_env)` | (B,) | element-wise: no index is summed, every cup cools on its own |
| $k_b t_t$ | `einsum('t,b->tb', t, k)` | (T, B) | outer product: every time with every cup, for the exact solution |
| $t_{1/2}$ | `log(2) / k` | (B,) | time for the temperature gap to halve |

The law says the cooling rate is proportional to the temperature gap to the room. The gap therefore shrinks by the same factor in equal times: it halves every $t_{1/2} = \ln 2/k$, whatever the starting temperature. The ODE is solved numerically with Runge–Kutta (RK4) and the dashed lines show the exact exponential for comparison. Real cups cool by convection, radiation and evaporation at once; $k$ lumps all of them into one number.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/thermodynamics/newton_cooling/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n, T_env = params["n_cups"], params["T_env"]
    k = tensors.cooling_constants(n, params["k_min"], params["k_max"], dev)  # (B,)
    T0 = tensors.start_temperatures(n, dev)  # (B,)

    times, temps = solve.solve(T0, k, T_env, params["t_max"])  # (T,), (T, B)
    exact = equation.exact_temperature(times, T0, k, T_env)  # (T, B)

    for b, t_half in enumerate(equation.half_life(k).tolist()):
        print(f"cup {b + 1}: k = {k[b]:.3f} /min   t½ = {t_half:5.1f} min   T(end) = {temps[-1, b]:5.1f} °C")
    print(f"largest difference RK4 vs exact: {(temps - exact).abs().max():.2e} °C")
    return plot.animate(times, temps, exact, k, T_env, out_dir / "newton_cooling.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/thermodynamics/newton_cooling/equation.py
r"""Physics → tensor translation.

Newton's law of cooling: an object loses heat at a rate proportional to how much hotter it is
than its surroundings,

    \frac{dT}{dt} = -k \, (T - T_{env})

For a batch of cups b = 1..B, each with its own cooling constant k_b, the index form is

    \dot T_b = -k_b \, (T_b - T_{env})

No index is repeated, so nothing is summed: it is a plain element-wise operation on the (B,) tensor.
The exact solution at times t_t is

    T_{tb} = T_{env} + (T_{0,b} - T_{env}) \, e^{-k_b t_t}

where k_b t_t (again no repeated index) is an outer product, einsum('t,b->tb').
The gap T - T_env halves every t_{1/2} = \ln 2 / k.
"""

import math

import torch


def cooling_rate(T: torch.Tensor, k: torch.Tensor, T_env: float) -> torch.Tensor:
    r"""\dot T_b = -k_b (T_b - T_env).   T, k: (B,) → (B,)"""
    return -k * (T - T_env)


def exact_temperature(t: torch.Tensor, T0: torch.Tensor, k: torch.Tensor, T_env: float) -> torch.Tensor:
    r"""T_{tb} = T_env + (T_{0,b} - T_env) e^{-k_b t_t}.   t: (T,), T0, k: (B,) → (T, B)"""
    kt = torch.einsum("t,b->tb", t, k)  # (T, B)  every time with every cup
    return T_env + (T0 - T_env) * torch.exp(-kt)  # T0 (B,) broadcasts over the time rows


def half_life(k: torch.Tensor) -> torch.Tensor:
    """Time for the temperature gap to halve: t½ = ln 2 / k.   (B,) → (B,)"""
    return math.log(2) / k


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/thermodynamics/newton_cooling/tensors.py
"""Build the concrete tensors: one starting temperature and one cooling constant per cup."""

import torch


def cooling_constants(n: int, k_min: float, k_max: float, device: torch.device) -> torch.Tensor:
    """k_b in 1/min, evenly spread from slow to fast cooling.  Shape (B,)."""
    if n == 1:
        return torch.tensor([k_min], device=device)
    return torch.linspace(k_min, k_max, n, device=device)


def start_temperatures(n: int, device: torch.device) -> torch.Tensor:
    """T_{0,b} in °C: the cups are poured between 95 °C and 70 °C.  Shape (B,)."""
    if n == 1:
        return torch.tensor([90.0], device=device)
    return torch.linspace(95.0, 70.0, n, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/thermodynamics/newton_cooling/solve.py
"""Integrate the batch of cooling ODEs with 4th-order Runge–Kutta."""

import torch

from problems._common.integrators import integrate

from .equation import cooling_rate


def solve(
    T0: torch.Tensor, k: torch.Tensor, T_env: float, t_max: float, dt: float = 0.1
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,) in minutes, temperatures (T, B) in °C).

    One RK4 step updates the whole (B,) tensor: every cup cools at the same time.
    """
    n_steps = int(round(t_max / dt))
    temps = integrate(lambda T, t: cooling_rate(T, k, T_env), T0, dt, n_steps)  # (T, B)
    times = torch.arange(n_steps + 1, device=T0.device) * dt
    return times, temps


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/thermodynamics/newton_cooling/plot.py
"""Animate the cups as thermometer bars next to their cooling curves T(t)."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(
    times: torch.Tensor, temps: torch.Tensor, exact: torch.Tensor, k: torch.Tensor, T_env: float, path: Path
) -> Path:
    t = times.cpu().numpy()
    T = temps.cpu().numpy()  # (T, B)
    T_exact = exact.cpu().numpy()  # (T, B)
    n_cups = T.shape[1]
    colors = [ACCENT[b % len(ACCENT)] for b in range(n_cups)]
    cmap = plt.get_cmap("coolwarm")
    T_low, T_high = T_env - 5, 100.0

    fig, (ax_bar, ax_T) = plt.subplots(1, 2, figsize=(7, 4), gridspec_kw={"width_ratios": [1, 1.6]})

    # Left: one "thermometer" bar per cup, coloured from blue (cold) to red (hot).
    cups = list(range(1, n_cups + 1))
    bars = ax_bar.bar(cups, T[0] - T_low, bottom=T_low, edgecolor=colors, linewidth=2)
    ax_bar.axhline(T_env, color="gray", ls="--", lw=1)
    ax_bar.set_ylim(T_low, T_high)
    ax_bar.set_xticks(cups)
    ax_bar.set_xlabel("cup")
    ax_bar.set_ylabel("T (°C)")
    clock = ax_bar.set_title("")

    # Right: the simulated curves grow over time; dashed lines are the exact exponentials.
    lines = []
    for b in range(n_cups):
        ax_T.plot(t, T_exact[:, b], color=colors[b], ls="--", lw=1, alpha=0.5)
        lines.append(ax_T.plot([], [], color=colors[b], lw=2, label=f"k = {k[b]:.3f}")[0])
    ax_T.axhline(T_env, color="gray", ls="--", lw=1, label="room")
    ax_T.set_xlim(0, t[-1])
    ax_T.set_ylim(T_low, T_high)
    ax_T.set_xlabel("t (min)")
    ax_T.set_title("solid: RK4    dashed: exact exponential", fontsize=9)
    ax_T.legend(fontsize=7, loc="upper right", title="1/min", title_fontsize=7)

    def update(step: int):
        for b, (bar, line) in enumerate(zip(bars, lines)):
            bar.set_height(T[step, b] - T_low)
            bar.set_facecolor(cmap((T[step, b] - T_low) / (T_high - T_low)))
            line.set_data(t[: step + 1], T[: step + 1, b])
        clock.set_text(f"t = {t[step]:.0f} min")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 90), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_cups': 5,
    'T_env': 20.0,
    'k_min': 0.03,
    'k_max': 0.15,
    't_max': 60.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.thermodynamics.newton_cooling")]:
    del sys.modules[name]
main = importlib.import_module("problems.thermodynamics.newton_cooling.main")
problem_dir = Path("problems/thermodynamics/newton_cooling")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
